In [65]:
from google.colab import files
import pandas as pd
import numpy as np




In [66]:
uploaded = files.upload()

Saving Data_Consolidada_Variables.xlsx to Data_Consolidada_Variables (2).xlsx


In [67]:


xlsx = pd.ExcelFile("Data_Consolidada_Variables.xlsx") # Lectura del archivo excel consolidado
xlsx.sheet_names


['CON_VAR', 'PRECIPITACION', 'TMAX', 'TMIN', 'TPROM', 'HR', 'RAD_SOL']

In [68]:
df = pd.read_excel("Data_Consolidada_Variables.xlsx", sheet_name="CON_VAR") # Lectura de la hoja del excel donde está todas las variables consolidadas
df.head()


,Fecha,Mes,Año,Día,PRECIPITACIÓN,TMAX,TMIN,TPROM,HR,RAD_SOL
0,2010-01-01,1,2010,1,0.0,24.2,NaN,NaN,NaN,3044.2
1,2010-01-02,1,2010,2,0.0,24.0,NaN,NaN,NaN,3103.0
2,2010-01-03,1,2010,3,0.0,24.6,NaN,NaN,NaN,2700.0
3,2010-01-04,1,2010,4,0.0,24.4,NaN,NaN,NaN,3634.0
4,2010-01-05,1,2010,5,0.0,25.4,NaN,NaN,NaN,4077.1


En esta etapa se realizó la normalización y estandarización de las columnas de fecha para garantizar la coherencia en el análisis. Las fechas presentaban diferentes formatos (por ejemplo, 2010-01-01 00:00 y 1/01/2010), lo que impedía su uso como clave en operaciones de búsqueda y consolidación. Para resolverlo, se convirtieron todas las entradas a un formato uniforme de tipo fecha, eliminando valores nulos y corrigiendo inconsistencias. Este proceso es fundamental para asegurar la integridad de los datos y evitar errores en cálculos posteriores.

In [69]:
df.info()
df.head(10)
df.tail(10)
df.describe(include='all')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5293 entries, 0 to 5292
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Fecha          5293 non-null   datetime64[ns]
 1   Mes            5293 non-null   int64         
 2   Año            5293 non-null   int64         
 3   Día            5293 non-null   int64         
 4   PRECIPITACIÓN  5293 non-null   float64       
 5   TMAX           3861 non-null   float64       
 6   TMIN           2023 non-null   float64       
 7   TPROM          3442 non-null   float64       
 8   HR             3466 non-null   float64       
 9   RAD_SOL        3478 non-null   float64       
dtypes: datetime64[ns](1), float64(6), int64(3)
memory usage: 413.6 KB


,Fecha,Mes,Año,Día,PRECIPITACIÓN,TMAX,TMIN,TPROM,HR,RAD_SOL
count,5293,5293.000000,5293.000000,5293.000000,5293.000000,3861.000000,2023.000000,3442.000000,3466.000000,3478.000000
mean,2017-05-13 07:41:57.211411456,6.590402,2016.859059,15.731721,3.950104,24.058948,21.751557,25.059413,77.951529,1300.712766
min,2010-01-01 00:00:00,1.000000,2010.000000,1.000000,0.000000,16.600000,17.100000,19.800000,0.000000,0.000000
25%,2013-08-16 00:00:00,4.000000,2013.000000,8.000000,0.000000,23.000000,20.500000,24.100000,76.000000,95.000000
50%,2017-03-31 00:00:00,7.000000,2017.000000,16.000000,0.000000,24.200000,21.500000,25.100000,86.000000,753.000000
75%,2020-11-14 00:00:00,10.000000,2020.000000,23.000000,3.000000,25.200000,22.850000,26.100000,94.000000,2275.275000
max,2024-12-31 00:00:00,12.000000,2024.000000,31.000000,189.000000,29.800000,28.600000,29.800000,110.000000,5921.600000
std,NaN,3.473417,4.327941,8.800156,10.431776,1.703476,1.702409,1.542898,27.471438,1356.711876


Columnas con NaN (faltantes reales)
TMAX → 1432 NaN
TMIN → 3270 NaN
TPROM → 1851 NaN
HR → 1827 NaN
RAD_SOL → 1815 NaN

In [70]:
#Vamos a revisar que columnas tiene mas faltante de datos
df.groupby("Año")["TMAX"].apply(lambda x: x.isna().sum())


,TMAX
Año,
2010,0
2011,16
2012,346
2013,365
2014,365
2015,148
2016,27
2017,39
2018,18


In [71]:

df.groupby("Año")["TMIN"].apply(lambda x: x.isna().sum())



,TMIN
Año,
2010,365
2011,351
2012,255
2013,265
2014,319
2015,228
2016,243
2017,255
2018,12


## Fórmula climatológica para TMAX

Si faltan datos durante meses o años, la imputación lineal NO funciona.

La estrategia recomendada es:

✔ Interpolación por día + ciclo estacional
𝑇
𝑀
𝐴
𝑋
_
𝑖
𝑚
𝑝
𝑢
𝑡
𝑒
𝑑
=
𝑇
𝑀
𝐴
𝑋
_
𝑐
𝑙
𝑖
𝑚
𝑎
𝑡
𝑜
𝑙
𝑜
𝑔
ı
ˊ
𝑎

𝑚
𝑒
𝑛
𝑠
𝑢
𝑎
𝑙
TMAX_imputed=TMAX_climatolog
ı
ˊ
a mensual

In [72]:
clima = df.groupby("Mes")["TMAX"].mean()
df.loc[df["TMAX"].isna(), "TMAX"] = df.loc[df["TMAX"].isna(), "Mes"].map(clima)
df["TMAX"] = df["TMAX"].rolling(window=7, min_periods=1, center=True).mean()
df["TMAX"].isna().sum()



np.int64(0)

## IMPUTACIÓN DE TMIN (la variable más incompleta)

Tu dataset muestra que TMIN está muy incompleta:

Tiene ~3270 NaN

Faltan varios años completos

Debemos reconstruirla usando TMAX (que ya está limpia)

La relación térmica en clima tropical es estable:

𝑇
𝑀
𝐼
𝑁
=
𝑇
𝑀
𝐴
𝑋
−
Δ
𝑇
TMIN=TMAX−ΔT

Donde:

ΔT típico = 6.0 a 7.0°C

Tu valor ajustado fue 6.5°C (muy correcto para Santander)

In [73]:
# Reconstrucción climatológica de TMIN basada en TMAX
df["TMIN"] = df["TMIN"].fillna(df["TMAX"] - 6.5)

# Suavizado para reducir saltos
df["TMIN"] = df["TMIN"].rolling(window=5, min_periods=1, center=True).mean()

df["TMIN"].isna().sum(), df["TMIN"].describe()



(np.int64(0),
 count    5293.000000
 mean       19.131088
 std         2.062524
 min        13.945714
 25%        17.528730
 50%        18.987891
 75%        20.740000
 max        26.000000
 Name: TMIN, dtype: float64)

## Usando los valores TMAX y TMIN vamos a recontruir los valores faltantes de TPROM y con eso garantizamos que no existan huecos

In [74]:
df["TPROM"] = (df["TMAX"] + df["TMIN"]) / 2
df["TPROM"].describe()


,TPROM
count,5293.000000
mean,21.599510
std,1.417559
min,17.190000
25%,20.689248
50%,21.615714
75%,22.548508
max,26.471429


## Limpieza de datos RAD_SOL

Se va realizar una mejora con los datos de la nasa para imputar los datos faltantes

In [75]:
df["RAD_SOL"].isna().sum(), df["RAD_SOL"].describe(), df.groupby("Año")["RAD_SOL"].count()


(np.int64(1815),
 count    3478.000000
 mean     1300.712766
 std      1356.711876
 min         0.000000
 25%        95.000000
 50%       753.000000
 75%      2275.275000
 max      5921.600000
 Name: RAD_SOL, dtype: float64,
 Año
 2010    362
 2011     91
 2012    365
 2013    365
 2014     99
 2015      0
 2016    316
 2017    339
 2018    322
 2019    365
 2020    366
 2021    184
 2022    304
 2023      0
 2024      0
 Name: RAD_SOL, dtype: int64)

In [76]:
# 1. Convertir valores imposibles a NaN (radiación > 4000 Wh/m2)
df.loc[df["RAD_SOL"] > 4000, "RAD_SOL"] = np.nan

# 2. Detectar valores mínimos irreales (muchos ceros)
df.loc[df["RAD_SOL"] < 10, "RAD_SOL"] = np.nan


In [77]:
#Descarga de informacion de la NASA

import requests
import pandas as pd

LAT = 6.876
LON = -73.409

url = "https://power.larc.nasa.gov/api/temporal/daily/point"

params = {
    "start": 20100101,
    "end": 20241231,
    "latitude": LAT,
    "longitude": LON,
    "community": "AG",
    "parameters": "ALLSKY_SFC_SW_DWN",
    "format": "JSON"
}

response = requests.get(url, params=params)
data = response.json()

# Convertir a dataframe
rad_nasa = pd.DataFrame.from_dict(data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"], orient="index")
rad_nasa.index = pd.to_datetime(rad_nasa.index)
rad_nasa.columns = ["RAD_SOL_NASA"]

rad_nasa.head()


,RAD_SOL_NASA
2010-01-01,21.92
2010-01-02,21.59
2010-01-03,22.92
2010-01-04,21.90
2010-01-05,22.44


In [78]:
# Se va imputar datos faltantes con los datos descargados de la nasa
# Unir datasets por fecha
df = df.merge(rad_nasa, left_on="Fecha", right_index=True, how="left")

# Si falta IDEAM, usar NASA
df["RAD_SOL"] = df["RAD_SOL"].fillna(df["RAD_SOL_NASA"])

# Drop columna NASA si quieres
df.drop(columns=["RAD_SOL_NASA"], inplace=True)


In [79]:
df["RAD_SOL"].isna().sum()


np.int64(0)

In [80]:
df["RAD_SOL"].describe()


,RAD_SOL
count,5293.000000
mean,715.424510
std,1061.388327
min,6.160000
25%,18.610000
50%,23.050000
75%,1007.000000
max,3994.000000


In [81]:
df["RAD_SOL"].describe(percentiles=[0.90, 0.95, 0.97, 0.99])


,RAD_SOL
count,5293.000000
mean,715.424510
std,1061.388327
min,6.160000
50%,23.050000
90%,2618.780000
95%,3157.500000
97%,3469.200000
99%,3809.448000
max,3994.000000


In [82]:
df["RAD_SOL"].value_counts().head(10)


,count
RAD_SOL,
636.0,22
639.0,19
366.0,17
817.0,16
545.0,15
455.0,14
94.0,14
459.0,13
547.0,12


In [83]:
# -------------------------------------------------
# 1. Copia de seguridad de la variable original
# -------------------------------------------------
df["RAD_SOL_original"] = df["RAD_SOL"]

# -------------------------------------------------
# 2. Invalidar valores físicamente imposibles
#    (umbral conservador: 1200 W/m²)
# -------------------------------------------------
df.loc[df["RAD_SOL"] > 1200, "RAD_SOL"] = None
df.loc[df["RAD_SOL"] < 0, "RAD_SOL"] = None

# -------------------------------------------------
# 3. Crear variable auxiliar de mes
# -------------------------------------------------
df["mes"] = df["Fecha"].dt.month

# -------------------------------------------------
# 4. Calcular promedio mensual con valores válidos
# -------------------------------------------------
rad_mes = df.groupby("mes")["RAD_SOL"].mean()

# -------------------------------------------------
# 5. Imputar valores faltantes usando promedio mensual
#    (manteniendo el nombre RAD_SOL)
# -------------------------------------------------
df["RAD_SOL"] = df["RAD_SOL"].fillna(
    df["mes"].map(rad_mes)
)

# -------------------------------------------------
# 6. (Opcional) Eliminar columna auxiliar de mes
# -------------------------------------------------
df.drop(columns=["mes"], inplace=True)


In [84]:
df["RAD_SOL"].describe(percentiles=[0.90, 0.95, 0.97, 0.99])

,RAD_SOL
count,5293.000000
mean,202.035847
std,280.864772
min,6.160000
50%,23.050000
90%,642.800000
95%,905.000000
97%,1001.000000
99%,1176.000000
max,1197.000000


## Vamos a imputar datos faltantes a HR (Humedad relativa)

Se va usar el modelo RandomForest para imputar datos faltantes

In [85]:
df["HR"].isna().sum(), df["HR"].describe()
df.groupby("Año")["HR"].mean()


,HR
Año,
2010,NaN
2011,94.014493
2012,85.696721
2013,83.612100
2014,87.324324
2015,76.632877
2016,85.612022
2017,87.590244
2018,84.590038


In [86]:
df.loc[(df["HR"] < 50), "HR"] = np.nan
df["HR"].isna().sum(), df.groupby("Año")["HR"].mean()



(np.int64(2178),
 Año
 2010          NaN
 2011    94.014493
 2012    85.696721
 2013    83.612100
 2014    87.324324
 2015    81.017391
 2016    85.612022
 2017    87.590244
 2018    84.590038
 2019    88.076696
 2020    91.254658
 2021    93.106195
 2022    89.865546
 2023    82.200000
 2024    85.520468
 Name: HR, dtype: float64)

In [87]:
# Tratamiento de datos mayor 100% HR
df["mes"] = df["Fecha"].dt.month

# promedio mensual válido
hr_mean_mes = (
    df[(df["HR"] >= 0) & (df["HR"] <= 100)]
    .groupby("mes")["HR"]
    .mean()
)

# reemplazar >100 usando promedio del mes
mask = (df["HR"] > 100) | (df["HR"] < 0)
df.loc[mask, "HR"] = df.loc[mask, "mes"].map(hr_mean_mes)


In [89]:
# =================================================
# IMPUTACIÓN DE HUMEDAD RELATIVA (HR)
# MODELO RANDOM FOREST - ESTRATEGIA 1
# =================================================

# -------------------------------------------------
# 1. Librerías
# -------------------------------------------------
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


# -------------------------------------------------
# 2. Selección de variables predictoras
#    (NO se agregan variables nuevas)
# -------------------------------------------------
features = [
    "TMAX",
    "TMIN",
    "TPROM",
    "PRECIPITACIÓN"
]


# -------------------------------------------------
# 3. Construcción del dataset de entrenamiento
#    (solo filas con HR válida)
# -------------------------------------------------
train_df = df.dropna(subset=["HR"] + features)

X_train = train_df[features]
y_train = train_df["HR"]


# -------------------------------------------------
# 4. Definición del modelo Random Forest
#    (ajuste fino de hiperparámetros)
# -------------------------------------------------
rf_model = RandomForestRegressor(
    n_estimators=800,        # más árboles = mayor estabilidad
    max_depth=15,            # complejidad moderada
    min_samples_leaf=8,      # reduce sobreajuste
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)


# -------------------------------------------------
# 5. Entrenamiento del modelo
# -------------------------------------------------
rf_model.fit(X_train, y_train)


# -------------------------------------------------
# 6. Evaluación del modelo (sobre datos conocidos)
# -------------------------------------------------
y_pred_train = rf_model.predict(X_train)

r2 = r2_score(y_train, y_pred_train)

mse = mean_squared_error(y_train, y_pred_train)
rmse = np.sqrt(mse)

mae = mean_absolute_error(y_train, y_pred_train)

print("Métricas del modelo Random Forest:")
print(f"R²   : {r2:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"MAE  : {mae:.3f}")


# -------------------------------------------------
# 7. Identificación de registros con HR faltante
# -------------------------------------------------
missing_df = df[df["HR"].isna()]
X_missing = missing_df[features]


# -------------------------------------------------
# 8. Imputación de HR en el DataFrame original
# -------------------------------------------------
df.loc[df["HR"].isna(), "HR"] = rf_model.predict(X_missing)


# -------------------------------------------------
# 9. Verificación final de la imputación
# -------------------------------------------------
print("\nVerificación final:")
print("Valores faltantes de HR:", df["HR"].isna().sum())
print("\nResumen estadístico de HR:")
print(df["HR"].describe())




Métricas del modelo Random Forest:
R²   : 0.402
RMSE : 7.600
MAE  : 6.003

Verificación final:
Valores faltantes de HR: 0

Resumen estadístico de HR:
count    5293.000000
mean       86.397159
std         8.041468
min        50.000000
25%        82.568128
50%        87.170466
75%        92.000000
max       100.000000
Name: HR, dtype: float64


In [90]:
# Confirmación de las imputaciones
df["HR"].isna().sum(), df.describe(), df.groupby("Año")["HR"].mean()


(np.int64(0),
                                Fecha          Mes          Año          Día  \
 count                           5293  5293.000000  5293.000000  5293.000000   
 mean   2017-05-13 07:41:57.211411456     6.590402  2016.859059    15.731721   
 min              2010-01-01 00:00:00     1.000000  2010.000000     1.000000   
 25%              2013-08-16 00:00:00     4.000000  2013.000000     8.000000   
 50%              2017-03-31 00:00:00     7.000000  2017.000000    16.000000   
 75%              2020-11-14 00:00:00    10.000000  2020.000000    23.000000   
 max              2024-12-31 00:00:00    12.000000  2024.000000    31.000000   
 std                              NaN     3.473417     4.327941     8.800156   
 
        PRECIPITACIÓN         TMAX         TMIN        TPROM           HR  \
 count    5293.000000  5293.000000  5293.000000  5293.000000  5293.000000   
 mean        3.950104    24.067931    19.131088    21.599510    86.397159   
 min         0.000000    20.37142

In [91]:
df["HR"].isna().sum(), df["HR"].describe()


(np.int64(0),
 count    5293.000000
 mean       86.397159
 std         8.041468
 min        50.000000
 25%        82.568128
 50%        87.170466
 75%        92.000000
 max       100.000000
 Name: HR, dtype: float64)

## Decarga de data limpia para usar

In [92]:
# Decarga de data limpia para usar
from google.colab import files

df.to_csv("Data_Limpia_2010_2024.csv",
          index=False,
          sep=";",
          decimal=",")



# Descargar al PC
files.download("Data_Limpia_2010_2024.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df.head()

In [ ]:
df.describe()